# 🎬 Novel Video Factory v4 — Kaggle Notebook
**Korean Manhwa-style video from novel scripts — 100% FREE**

## Features
- 🧠 **LLM**: Groq free-tier (llama-3.3-70b) or Ollama local (qwen2.5:7b)
- 🎨 **Images**: Animagine-XL 3.1 (best free manhwa model, no API key)
- 🔊 **Audio**: Microsoft Edge TTS (free, natural voices)
- 🎥 **Video**: 10-minute clips → stitched into 2-hour final video
- ♻️ **Resume**: Re-run any cell after disconnect — nothing is lost

## Setup
1. Add your novel .txt file to `projects/novel/input/`
2. (Optional) Add `GROQ_API_KEY` to Kaggle Secrets for better LLM
3. Run cells top-to-bottom. Each stage saves progress automatically.


In [ ]:
# ── Cell 1: Setup (run once per session) ──────────────────────────────────
!bash kaggle_setup.sh

In [ ]:
# ── Cell 2: Config ────────────────────────────────────────────────────────
import os

PROJECT_NAME = "novel"          # Must match your folder in projects/
NOVEL_FILE   = None             # Set to path if importing a new file
                                # e.g. "/kaggle/input/my-novel/chapter1.txt"

# Optional: set Groq key here if not using Kaggle Secrets
# os.environ["GROQ_API_KEY"] = "gsk_..."

# Try to load from Kaggle Secrets automatically
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["GROQ_API_KEY"] = UserSecretsClient().get_secret("GROQ_API_KEY")
    print("✓ GROQ_API_KEY loaded from Kaggle Secrets")
except Exception:
    print("ℹ️  No GROQ_API_KEY found — using Ollama local LLM")

print(f"Project: {PROJECT_NAME}")

In [ ]:
# ── Cell 3: Stage 1 — Translation ─────────────────────────────────────────
# Smart skip: if your novel is already in English, this stage does nothing
import sys
sys.path.insert(0, os.getcwd())

from core.orchestrator import UnifiedPipeline
pipeline = UnifiedPipeline(PROJECT_NAME)

if NOVEL_FILE:
    pipeline.import_source(NOVEL_FILE)

pipeline.stage_translate()
print("\n✅ Translation done!")

In [ ]:
# ── Cell 4: Stage 2 — Memory Extraction ───────────────────────────────────
# Extracts characters, locations, lore from the novel
pipeline.stage_memory()
print("\n✅ Memory extraction done!")

# Show what was extracted
chars = pipeline.memory_db.get_all_characters()
locs  = pipeline.memory_db.get_all_locations()
print(f"   Characters: {[c['canonical_name'] for c in chars]}")
print(f"   Locations:  {[l['canonical_name'] for l in locs]}")

In [ ]:
# ── Cell 5: Stage 3 — Character Sheets ────────────────────────────────────
# Generates 6 reference poses per character (for IP-Adapter consistency)
# This cell can take a while — ~6 images per character
pipeline.stage_character_sheets()
print("\n✅ Character sheets done!")

In [ ]:
# ── Cell 6: Stage 4 — Visual Planning ─────────────────────────────────────
# Breaks the novel into scenes and builds image prompts
pipeline.stage_visual_planning()
print("\n✅ Visual planning done!")

# Show clip summary
import json
clips_path = f"projects/{PROJECT_NAME}/output/clips.json"
if os.path.exists(clips_path):
    with open(clips_path) as f:
        clips = json.load(f)
    total_scenes = sum(len(c.get('shots', [])) for c in clips)
    print(f"   {len(clips)} clips | {total_scenes} scenes total")
    for c in clips:
        print(f"   {c['clip_id']}: {c['scene_count']} scenes")

In [ ]:
# ── Cell 7: Stage 5 — Image Generation ────────────────────────────────────
# Generates all manhwa-style images (longest stage — ~6-8s per image)
# SAFE TO INTERRUPT AND RESUME: already-generated images are skipped
pipeline.stage_generation()
print("\n✅ Image generation done!")

In [ ]:
# ── Cell 8: Stage 6 — Audio (TTS) ─────────────────────────────────────────
# Generates narration audio using Microsoft Edge TTS (FREE)
pipeline.stage_audio()
print("\n✅ Audio generation done!")

In [ ]:
# ── Cell 9: Stage 7 — Video Assembly ──────────────────────────────────────
# Assembles images + audio into 10-minute MP4 clips
# Each clip is saved independently — safe to interrupt and resume
pipeline.stage_video()
print("\n✅ Video assembly done!")

# List output clips
videos_dir = f"projects/{PROJECT_NAME}/output/videos"
if os.path.exists(videos_dir):
    for f in sorted(os.listdir(videos_dir)):
        if f.endswith('.mp4'):
            size_mb = os.path.getsize(os.path.join(videos_dir, f)) / (1024*1024)
            print(f"   {f}: {size_mb:.1f} MB")

In [ ]:
# ── Cell 10: Export / Download ─────────────────────────────────────────────
pipeline.stage_export()

# Option A: Download final video directly from Kaggle
final_path = f"projects/{PROJECT_NAME}/output/videos/final_video.mp4"
if os.path.exists(final_path):
    from IPython.display import FileLink
    print("Download your video:")
    display(FileLink(final_path))
else:
    print("Final video not found — check stage_video logs above")

In [ ]:
# ── Cell 11: (Optional) Upload to Google Drive ─────────────────────────────
# Set gdrive.enabled: true and gdrive.folder_id in config/default.yaml first

UPLOAD_TO_DRIVE = False   # Set to True to enable

if UPLOAD_TO_DRIVE:
    from google.colab import auth
    auth.authenticate_user()
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaFileUpload
    import google.auth

    creds, _ = google.auth.default()
    drive = build('drive', 'v3', credentials=creds)

    folder_id = pipeline.config.get('gdrive.folder_id', '')
    if not folder_id:
        print("Set gdrive.folder_id in config/default.yaml first")
    else:
        media = MediaFileUpload(final_path, mimetype='video/mp4', resumable=True)
        file = drive.files().create(
            body={'name': 'novel_video.mp4', 'parents': [folder_id]},
            media_body=media, fields='id'
        ).execute()
        print(f"✅ Uploaded to Drive: https://drive.google.com/file/d/{file['id']}")

In [ ]:
# ── Cell 12: Preview a sample image ───────────────────────────────────────
import glob
from IPython.display import Image, display

images_dir = f"projects/{PROJECT_NAME}/output/images"
sample_images = sorted(glob.glob(f"{images_dir}/*.png"))[:6]

if sample_images:
    print(f"Showing {len(sample_images)} sample generated images:")
    for img_path in sample_images:
        display(Image(img_path, width=400))
        print(os.path.basename(img_path))
else:
    print("No images yet — run stage_generation first")

In [ ]:
# ── Cell 13: Run FULL pipeline (all stages at once) ────────────────────────
# Alternative to running stages individually
# Use this if you want to run everything in one go

# pipeline.run_all(source_file=NOVEL_FILE)
# print("\n🎉 Full pipeline complete!")